# N-1 Fine tuning task for FedOPF

* Create N-1 contingencies: number & indicies of cut off lines and generators
    * For the dataset, we only need to generate load because this is unsupervised way 
* Test on the zero-shot (i.e., just inference by using pretrained client/server/global model)
    * **In 162 case, fine-tuning is only needed for branch contingency**

In [1]:
# !pip install torch_geometric
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.0+cu121.html

# !pip install pypower
# !pip install pyrlu
# # !pip install conflictfree

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
import os
# os.environ["PYTORCH_LINALG_WARNINGS"] = "0"
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "0"
# os.environ["CUDA_VISIBLE_DEVICES"] = '0, 1, 2, 3'

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', device)  # 출력결과: cuda
print('Count of using GPUs:', torch.cuda.device_count())   #출력결과: 1 (GPU #2 한개 사용하므로)
print('Current cuda device:', torch.cuda.current_device())  # 출력결과: 2 (GPU #2 의미)

Device: cuda
Count of using GPUs: 1
Current cuda device: 0


In [4]:
from utils.utils162_graphlde import ACOPFProblem
grid_data_path = './data/pglib_opf_case162.mat'
grid = ACOPFProblem(grid_filename=grid_data_path)

/global/u1/k/kjsong/FedOPF-APPFL/fine-tuning-task/n-1/case162/utils/utils162_graphlde.py:88: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.slackva = torch.tensor([np.deg2rad(self.ppc['bus'][self.slack, idx_bus.VA])],


In [5]:
component="branch" # "gen" or "branch"
num_cases = 2
contingencies = grid.check_contingencies(component=component, num_cases=num_cases)

contingencies['contingencies'] = [x for x in contingencies['contingencies'] if x]
acopf_contingencies = []
for c in range(contingencies['num_cases']):
    acopf_contingencies += [ACOPFProblem(grid_filename=grid_data_path, contingencies=contingencies['contingencies'][c], component=component)] # update the grid components' values from modified ppc data based on the contingencies 

    # Put all variables in "data" to the cuda.
    acopf_contingencies[-1]._device = device
    for attr in dir(acopf_contingencies[-1]):
        var = getattr(acopf_contingencies[-1], attr)
        if not callable(var) and not attr.startswith("__") and torch.is_tensor(var):
            try:
                setattr(acopf_contingencies[-1], attr, var.to(device))
            except AttributeError:
                pass


12
283

Converged in 0.71 seconds
Objective Function Value = 108088.94 $/hr
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses            162     Total Gen Capacity   11032.0       -2784.0 to 4118.2
Generators        12     On-line Capacity     11032.0       -2784.0 to 4118.2
Committed Gens    12     Generation (actual)   7401.7            1047.8
Loads            113     Load                  7239.1            1174.6
  Fixed          113       Fixed               7239.1            1174.6
  Dispatchable     0       Dispatchable           0.0 of 0.0        0.0
Shunts            34     Shunt (inj)             -0.0            -203.3
Branches         283     Losses (I^2 * Z)       162.66           1952.29
Transformers      91     Branch Charging (inj)     -             2282.4
Inter-ties         0     To

In [7]:
contingencies

{'contingencies': [{'branch': 275}, {'branch': 141}], 'num_cases': 2}

In [ ]:
from pypower import idx_bus, idx_gen, idx_brch
import numpy as np
import torch_geometric
from torch_geometric.data import Data 
from models.Edge_GNN_solver import Edge_GNNSolver
import copy

# Load data generation (unsupervised)
pd_base = grid.ppc['bus'][:,idx_bus.PD] / grid.baseMVA.item()
qd_base = grid.ppc['bus'][:,idx_bus.QD] / grid.baseMVA.item()
num_data = 200
perturb = (1.0 + np.random.uniform(-0.2, 0.05, size=(num_data, grid.nbus)))

# data sample from uniform distribution
pd_samples = torch.from_numpy(pd_base * perturb).float()
qd_samples = torch.from_numpy(qd_base * perturb).float()
demand = pd_samples + 1j*qd_samples

In [8]:
# Graph data representation
graph_dataset_contingency = []
graph_dataset_contingency_statics = []

# Model config
train_config = {
    'probType': 'acopf',
    'useCompl': True, # boolean type: whether to use completion (DC3, DeepLDE 기술)

    # GNN model parameters
    'n_gnn_layers': 3,
    'nfeature_dim': 2, # input dim
    'efeature_dim': 4, # edge feature dim 
    'hidden_dim': 40,
    'dropout_rate': 0.1,
    'K': 10, # 6 # only for TAGConv or ChebConv and GATConv (as multi-head)

    # DeepLDE hyperparameters
    'epochs': 40, # 10 (GPU RTX 4090), 8 (GPU A100)
    'batchSize': 5, # 6, (8) (GPU RTX 4090; GPU 다운 에러 발생..), 16 (GPU A100)
    'lr': 1e-2, # 1e-3
    'lr_w': 1e-2, # 1e-3
    'weight_decay': 1e-3, # 1e-5,

    # LDF parameters
    'rho_init': 0.1, # 0.01  
    's_init': 0.1,
    'p_iter_max': 10,
    'warmup_iter': 40, # 20, # 0 for non-warmup start cases
    'corrEps': 1e-4, # float type: correction procedure tolerance
}

# Models
contingency_models = []

for case in range(len(acopf_contingencies)):
    node_feat_x = np.zeros((demand.shape[0], acopf_contingencies[case].nbus, 2))
    node_mask_x = np.zeros((demand.shape[0], acopf_contingencies[case].nbus))
    for s in range(node_feat_x.shape[0]):
        node_feat_x[s,:,0] = np.real(demand[s,:])
        node_feat_x[s,:,1] = np.imag(demand[s,:])
        node_mask_x[s,acopf_contingencies[case].spv] = 1
        # node_mask_x[s,self.pv] = 1
    
    edge_feat_x = np.zeros((demand.shape[0], acopf_contingencies[case].nl, 6))
    for s in range(edge_feat_x.shape[0]):
        edge_feat_x[s,:,0] = acopf_contingencies[case].ppc['branch'][:, idx_brch.F_BUS]
        edge_feat_x[s,:,1] = acopf_contingencies[case].ppc['branch'][:, idx_brch.T_BUS]
        edge_feat_x[s,:,2] = acopf_contingencies[case].ppc['branch'][:, idx_brch.BR_R]
        edge_feat_x[s,:,3] = acopf_contingencies[case].ppc['branch'][:, idx_brch.BR_X]
        edge_feat_x[s,:,4] = acopf_contingencies[case].ppc['branch'][:, idx_brch.BR_B]
        
        br_ratea_feat = (acopf_contingencies[case].ppc['branch'][:, idx_brch.RATE_A] / acopf_contingencies[case].baseMVA)
        # br_ratea_feat[br_ratea_feat == 0] = np.Inf
        edge_feat_x[s,:,5] = br_ratea_feat

    node_feat_x = torch.tensor(node_feat_x[:,:,:], dtype=torch.get_default_dtype())
    node_mask_x = torch.tensor(node_mask_x[:,:], dtype=torch.get_default_dtype())
    edge_feat_x = torch.tensor(edge_feat_x[:,:,:], dtype=torch.get_default_dtype())
    pyg_data_list = []
    for i in range(demand.shape[0]):
        pyg_data_list += [Data(
            x=node_feat_x[i,:,:],
            node_mask=node_mask_x[i,:].to(torch.bool),
            edge_index=edge_feat_x[i, :, 0:2].T.to(torch.long),
            edge_attr=edge_feat_x[i, :, 2:],) # R,X,B,RateA
            ]
    
    graph_dataset_contingency += [pyg_data_list]
    
    node_dataset = torch.zeros((acopf_contingencies[case].nbus*len(pyg_data_list),2))
    edge_dataset = torch.zeros((acopf_contingencies[case].nl*len(pyg_data_list),4))
    for (i,data) in enumerate(pyg_data_list):
        node_dataset[acopf_contingencies[case].nbus*i:acopf_contingencies[case].nbus*(i+1),:] = data.x
        edge_dataset[acopf_contingencies[case].nl*i:acopf_contingencies[case].nl*(i+1),:] = data.edge_attr
    
    node_means = node_dataset.mean(axis=0, keepdims = True)
    node_stds = node_dataset.std(axis=0, keepdims = True)
    edge_means = edge_dataset.mean(axis=0, keepdims = True)
    edge_stds = edge_dataset.std(axis=0, keepdims = True)

    graph_dataset_contingency_statics += [[node_means.to(dtype=torch.get_default_dtype()),
                                          node_stds.to(dtype=torch.get_default_dtype()),
                                          edge_means.to(dtype=torch.get_default_dtype()),
                                          edge_stds.to(dtype=torch.get_default_dtype())]]

    data_loader = torch_geometric.loader.DataLoader(pyg_data_list, batch_size=1, shuffle=False, drop_last=True)
    solver_net = Edge_GNNSolver(acopf_contingencies[case], train_config)
    contingency_models += [solver_net]

    print(solver_net)

Edge_GNNSolver(
  (layers): ModuleList(
    (0): EdgeAggregation()
    (1): TransformerConv(120, 40, heads=10)
    (2): EdgeAggregation()
    (3): TransformerConv(120, 40, heads=10)
    (4): EdgeAggregation()
    (5): TransformerConv(120, 40, heads=10)
  )
  (flatten): Linear(in_features=1440, out_features=23, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
Edge_GNNSolver(
  (layers): ModuleList(
    (0): EdgeAggregation()
    (1): TransformerConv(120, 40, heads=10)
    (2): EdgeAggregation()
    (3): TransformerConv(120, 40, heads=10)
    (4): EdgeAggregation()
    (5): TransformerConv(120, 40, heads=10)
  )
  (flatten): Linear(in_features=1440, out_features=23, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


/tmp/ipykernel_375096/483345036.py:57: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  node_feat_x[s,:,0] = np.real(demand[s,:])
/tmp/ipykernel_375096/483345036.py:58: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  node_feat_x[s,:,1] = np.imag(demand[s,:])


* Load pretrained GraphOPF model

In [9]:
pretrained_client = torch.load("./models/pretrained_models/client/checkpoint_Client7.pth", weights_only=False, map_location=device)
# pretrained_server = torch.load("./models/pretrained_models/server/checkpoint_Server_0.pth", weights_only=False, map_location=device)
# pretrained_global = torch.load("./models/pretrained_models/global/checkpoint_Global.pth", weights_only=False, map_location=device)

In [10]:
for contingency_model in contingency_models:
    if component == 'gen':
        print("Generator contingency")
        # Dimension reduction for N-1 contingency
        new_row = contingency_model.flatten.weight.shape[0] 
        new_col = contingency_model.flatten.weight.shape[1]
        old_row = pretrained_client['flatten.weight'].shape[0]
        old_col = pretrained_client['flatten.weight'].shape[1]
    
        pretrained_flatten_weight = copy.deepcopy(pretrained_client['flatten.weight'])
        pretrained_flatten_bias = copy.deepcopy(pretrained_client['flatten.bias'])
        
        # Way 1
        row_norm = torch.norm(pretrained_flatten_weight, dim=1)
        rows_removed = torch.argsort(row_norm)[:old_row - new_row]
        col_norm = torch.norm(pretrained_flatten_weight, dim=0)
        cols_removed = torch.argsort(col_norm)[:old_col - new_col]
        
        row_mask = torch.ones(pretrained_flatten_weight.size(0), dtype=torch.bool, device=device)
        row_mask[rows_removed] = False
        col_mask = torch.ones(pretrained_flatten_weight.size(1), dtype=torch.bool, device=device)
        col_mask[cols_removed] = False

        shrinked_weight = pretrained_flatten_weight[row_mask][:, col_mask]
        shrinked_bias = pretrained_flatten_bias[row_mask] # corresponding to the flatten.weight

        # # Way 2
        # shrinked_weight = pretrained_flatten_weight[(old_row - new_row):, (old_col - new_col):]
        # shrinked_bias = pretrained_flatten_bias[(old_row - new_row):] # corresponding to the flatten.weight

        ### Transfer the weights from the pretrained GraphOPF
        global_pretrained_state_dict = pretrained_client
        target_state_dict = contingency_model.state_dict()
        
        layer_names = list(target_state_dict.keys())
        for name in layer_names[:-2]:
            if name in target_state_dict.keys() and name in global_pretrained_state_dict.keys():
                # print(name)
                target_state_dict[name] = global_pretrained_state_dict[name].clone()
            
        for name in layer_names[-2:]:
            if 'weight' in name:
                # print('weight load')
                target_state_dict[name] = shrinked_weight
            elif 'bias' in name:
                # print('bias load')
                target_state_dict[name] = shrinked_bias
    
    elif component == 'branch':
        print("Branch contingency")
        ### Transfer the weights from the pretrained GraphOPF
        global_pretrained_state_dict = pretrained_client
        target_state_dict = contingency_model.state_dict()

        layer_names = list(target_state_dict.keys())
        for name in layer_names[:]:
            if name in target_state_dict.keys() and name in global_pretrained_state_dict.keys():
                # print(name)
                target_state_dict[name] = global_pretrained_state_dict[name].clone()

    contingency_model.load_state_dict(target_state_dict)

    # Freezing the half-GNN layers
    for i, (name, param) in enumerate(contingency_model.named_parameters()):
        if i <= 32: # 5, 10, 21
            # print(name)
            param.requires_grad = False
        else:
            print(name)
            param.requires_grad = True

Branch contingency
flatten.weight
flatten.bias
Branch contingency
flatten.weight
flatten.bias


* Fine-tuning (for generator contingency)

In [47]:
import torch.optim as optim
import time
from utils.loss_fn_graphlde import total_loss, ineq_violation
from utils.log import dict_agg

# Set the linear algebra solver(?)
torch._C._set_linalg_preferred_backend(torch._C._LinalgBackend.Cusolver) # torch._C._LinalgBackend.Magma, torch._C._LinalgBackend.Cusolver
# torch._C._get_linalg_preferred_backend() 

# stats_contingency = []
for case in range(len(acopf_contingencies)):
# for case in range(1):
    print("Contingency ", case)
    stats = {}
    
    # NOTE: LDF parameters.
    LagM_sp_gen = torch.ones(1, 2).to(device) # shape: (1, num_inequalities)
    LagM_gen = torch.ones(1, 2*acopf_contingencies[case].ng).to(device) # shape: (1, num_inequalities)
    LagM_bus = torch.ones(1, 2*acopf_contingencies[case].nbus).to(device) # shape: (1, num_inequalities)
    LagM_line = torch.ones(1, 2*acopf_contingencies[case].nl).to(device) # shape: (1, num_inequalities)
    
    warmup_iter = train_config["warmup_iter"] # the warmup period: the NN is trained with an additional inner iteration before the first outer iteration.
    rho_init = train_config["rho_init"]
    
    rho = rho_init
    
    rho_iter = 0
    s_iter = 0
    
    p_iter_max = train_config["p_iter_max"]
    p_iter_max_sum = p_iter_max
    
    d = 0 # 0 for static case otherwise use 5"
    beta = 0 # 0.001 # 0 for static case otherwise use 1
    
    lr_w = train_config["lr_w"] # 이거 증가해도 되지 않을지?
    lr = train_config["lr"]
    print_interval = 1
    eps_converge = 1e-4
    
    contingency_models[case] = contingency_models[case].to(device)
    train_loader = torch_geometric.loader.DataLoader(graph_dataset_contingency[case][:20], batch_size=train_config['batchSize'], shuffle=True, drop_last=True)
    node_means, node_stds, edge_means, edge_stds = graph_dataset_contingency_statics[case][0], graph_dataset_contingency_statics[case][1], graph_dataset_contingency_statics[case][2], graph_dataset_contingency_statics[case][3]
    n_means = node_means.to(device)
    n_stds = node_stds.to(device)
    e_means = edge_means.to(device)
    e_stds = edge_stds.to(device)
    
    train_loss_list = []
    train_start_time = time.time()
    for i in range(train_config["epochs"]):
        epoch_stats = {}
        ################### TRAINING PHASE ###################
        solver_net.train()
        if i<warmup_iter:
            ######### WARM-UP PERIOD #########
            if i == 0:
                solver_opt = optim.Adam(contingency_models[case].parameters(), lr=lr_w) # this will be reinitalized after warmup stage
                print("Warmup start!")
    
            for Xtrain in train_loader:
                Xtrain = Xtrain.to(device)
                # start_time = time.time()
                solver_opt.zero_grad()
    
                # DNN + NR (equality constraint)
                Yhat_train = contingency_models[case](Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
    
                # train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)
                train_loss, train_obj, ineq_dist, eq_resid = total_loss(acopf_contingencies[case], Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)
    
                train_loss.sum().backward()
                solver_opt.step()
    
                dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())
    
                ineq_p_g = ineq_dist[:,:2]
                ineq_q_g = ineq_dist[:,2:2+2*acopf_contingencies[case].ng]
                ineq_v_m = ineq_dist[:,2+2*acopf_contingencies[case].ng:2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus]
                ineq_line_l = ineq_dist[:,2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus:]
    
                dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())
    
                dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())
    
        else:
            if i == warmup_iter:
                print("Warmup ended!")
                # data.ref_freedom = False
    
                # ineq_lag_flag = True # for the first warmup end epoch, consider lag. multipliers update of ineq.
            # elif (i - warmup_iter)%ineq_lag_flag_trigger == 0:
            #     # print("Doing test... continue this case..")
            #     print("consider lagrangian multipliers update for ineq. constraints!")
            #     ineq_lag_flag = True
            # else:
            #     ineq_lag_flag = False
    
            # for every (updated) p_iter_max_sum time.
            if (i - warmup_iter)%p_iter_max_sum == 0:
                ######### Outer Interation: calculate step size of lagrangian multipliers update #########
                if i> warmup_iter:
                    print("current epoch %d || p_iter_max updated : %d -> %d" %(i, p_iter_max, p_iter_max + d))
                    # s = s_init * (1/(1+beta*(s_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                    # s_iter += 1
                    # print("mu iter updated : %d -> %d" %(s_iter-1, s_iter))
    
                    rho = rho_init * (1/(1+beta*(rho_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                    rho_iter += 1
                    print("rho iter updated : %d -> %d" %(rho_iter-1, rho_iter))
                    p_iter_max = p_iter_max + d
                    p_iter_max_sum += p_iter_max
    
                with torch.no_grad():
                    print("Lambda updated at %d epoch" %i)
                    contingency_models[case].eval()
                    for Xtrain in train_loader:
                        Xtrain = Xtrain.to(device)
                        #solver_opt.zero_grad()
                        Yhat_train = contingency_models[case](Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                        # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
    
                        # LagM += rho*ineq_violation(data, Xtrain.x, Yhat_train) # Update the lagrangian multiplier.
                        LagM_sp_gen += rho*ineq_violation(acopf_contingencies[case], Xtrain.x, Yhat_train)[:2] # Update the lagrangian multiplier.
                        LagM_gen += rho*ineq_violation(acopf_contingencies[case], Xtrain.x, Yhat_train)[2:2+2*acopf_contingencies[case].ng] # Update the lagrangian multiplier.
                        LagM_bus += rho*ineq_violation(acopf_contingencies[case], Xtrain.x, Yhat_train)[2+2*acopf_contingencies[case].ng:2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus] # Update the lagrangian multiplier.
                        LagM_line += rho*ineq_violation(acopf_contingencies[case], Xtrain.x, Yhat_train)[2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus:] # Update the lagrangian multiplier.
                
                ## Does this stablize the learning process much better?
                LagM_sp_gen = (LagM_sp_gen/len(train_loader))
                LagM_gen = (LagM_gen/len(train_loader))
                LagM_bus = (LagM_bus/len(train_loader))
                LagM_line = (LagM_line/len(train_loader))
    
                # solver_opt = optim.Adam(contingency_models[0].parameters(), lr = lr)
    
            contingency_models[case].train()
            for Xtrain in train_loader:
                Xtrain = Xtrain.to(device)
                solver_opt.zero_grad()
                Yhat_train = contingency_models[case](Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
    
                # train_loss, obj_train, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier (1, num_inequalities)
                train_loss, train_obj, ineq_dist, eq_resid = total_loss(acopf_contingencies[case], Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)
    
                train_loss.sum().backward()
                solver_opt.step()
    
                dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())
    
                ineq_p_g = ineq_dist[:,:2]
                ineq_q_g = ineq_dist[:,2:2+2*acopf_contingencies[case].ng]
                ineq_v_m = ineq_dist[:,2+2*acopf_contingencies[case].ng:2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus]
                ineq_line_l = ineq_dist[:,2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus:]
    
                dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())
    
                dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
                dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())
    
        if (i == 0) or (i%print_interval == 0):
            print(
                'Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                    i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                    np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                    np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))
        
        train_loss_list.append(np.mean(epoch_stats['train_loss']))
        # valid_eval_list.append(np.mean(epoch_stats['valid_eval']))
    # stats_contingency += []
    print("\n")
train_end_time = time.time()
total_train_time = train_end_time - train_start_time

Contingency  0
Warmup start!
Epoch 0: train loss 284.2203, train obj 11.1255, ineq max 87.1426, ineq mean 0.4391, ineq p_g num viol 1.0000, ineq q_g num viol 5.3000, ineq v_m num viol 1.0000, ineq line_theraml num viol 6.7500, eq max 0.0039, eq mean 0.0002
Epoch 1: train loss 136.6092, train obj 11.1515, ineq max 45.6682, ineq mean 0.2017, ineq p_g num viol 1.0000, ineq q_g num viol 4.9000, ineq v_m num viol 3.4500, ineq line_theraml num viol 5.7000, eq max 0.0045, eq mean 0.0002
Epoch 2: train loss 71.7393, train obj 11.1689, ineq max 21.6128, ineq mean 0.0974, ineq p_g num viol 1.0000, ineq q_g num viol 4.8500, ineq v_m num viol 5.9000, ineq line_theraml num viol 4.5000, eq max 0.0054, eq mean 0.0002
Epoch 3: train loss 49.8258, train obj 11.1810, ineq max 14.2181, ineq mean 0.0621, ineq p_g num viol 1.0000, ineq q_g num viol 4.0000, ineq v_m num viol 7.8500, ineq line_theraml num viol 3.5000, eq max 0.0062, eq mean 0.0003
Epoch 4: train loss 43.5559, train obj 11.1892, ineq max 13.4

In [48]:
total_train_time

4.135668516159058

* Evaluation (zero-shot)

In [11]:
from pypower.api import makeYbus
import time
from utils.loss_fn_graphlde import total_loss, ineq_violation
from utils.log import dict_agg

test_stats_contingency = []
for case in range(len(acopf_contingencies)):
    print("Contingency ", case)
    Ybus, Yf, Yt = makeYbus(acopf_contingencies[case].baseMVA, acopf_contingencies[case].ppc['bus'], acopf_contingencies[case].ppc['branch'])
    # branch thermal limit information
    flow_max = (acopf_contingencies[case].ppc['branch'][:, 5] / acopf_contingencies[case].baseMVA)**2
    flow_max[flow_max == 0] = np.inf
    flow_max = torch.tensor(flow_max, dtype=torch.float32).to(device)
    
    node_means, node_stds, edge_means, edge_stds = graph_dataset_contingency_statics[case][0], graph_dataset_contingency_statics[case][1], graph_dataset_contingency_statics[case][2], graph_dataset_contingency_statics[case][3]
    n_means = node_means.to(device)
    n_stds = node_stds.to(device)
    e_means = edge_means.to(device)
    e_stds = edge_stds.to(device)
    
    test_loader = torch_geometric.loader.DataLoader(graph_dataset_contingency[case][100:], batch_size=1, shuffle=False, drop_last=True)
    contingency_models[case] = contingency_models[case].to(device)
    contingency_models[case].eval()
    test_stats = {}
    # test_stats["Contingency"] = case
    test_eps_converge = 1e-4
    
    LagM_sp_g = torch.ones(1, 2).to(device) # shape: (1, num_inequalities)
    LagM_q_g = torch.ones(1, 2*acopf_contingencies[case].ng).to(device) # shape: (1, num_inequalities)
    LagM_v_m = torch.ones(1, 2*acopf_contingencies[case].nbus).to(device) # shape: (1, num_inequalities)
    LagM_line_l = torch.ones(1, 2*acopf_contingencies[case].nl).to(device) # shape: (1, num_inequalities)
    
    solve_time = []
    for (i, Xtest) in enumerate(test_loader):
        Xtest = Xtest.to(device)
    
        start_time = time.time()
        Y = contingency_models[case](Xtest, n_means, n_stds, e_means, e_stds)
        end_time = time.time()
    
        solve_time += [end_time - start_time]
    
        ## line thermal limit
        pg, qg, vm, va = acopf_contingencies[case].get_yvars(Y)
        vr = vm*torch.cos(va)
        vi = vm*torch.sin(va)
        vz = torch.complex(vr, vi) # complex voltage
    
        # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
        If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(device) @ vz.T
        It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(device) @ vz.T
    
        # Calculate the apparent power S
        Sf = vz[:,acopf_contingencies[case].ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
        St = vz[:,acopf_contingencies[case].ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
        Sff = Sf * torch.conj(Sf)
        Stt = St * torch.conj(St)
    
        # calculate the line thermal limit constraints violation
        diff_Sf = Sff.real - flow_max
        diff_St = Stt.real - flow_max
        # diff_Sf[torch.clamp(diff_Sf, 0) != 0]
    
        line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
        line_limit_vio_St = torch.clamp(diff_St, 0)
        ###########################################
    
        # test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM)
        test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(acopf_contingencies[case], Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)
    
        dict_agg(test_stats, 'time', end_time - start_time, op='sum')
    
        test_ineq_p_g = torch.cat([pg - acopf_contingencies[case].pmax, acopf_contingencies[case].pmin - pg], dim=1)
        test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(device)
        test_ineq_q_g = test_ineq_dist[:,2:2+2*acopf_contingencies[case].ng]
        test_ineq_v_m = test_ineq_dist[:,2+2*acopf_contingencies[case].ng:2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus]
        test_ineq_line_l = test_ineq_dist[:,2+2*acopf_contingencies[case].ng+2*acopf_contingencies[case].nbus:]
    
        dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
        # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())
    
        dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())
    
        dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())
    
        pg_rate_torch = (((pg <= acopf_contingencies[case].pmax) & (pg >= acopf_contingencies[case].pmin)).sum()/acopf_contingencies[case].ng)*100
        qg_rate_torch = (((qg <= acopf_contingencies[case].qmax) & (qg >= acopf_contingencies[case].qmin)).sum()/acopf_contingencies[case].ng)*100
        dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
        dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
        # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
        # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())
    
        v_rate_torch = (((vm <= acopf_contingencies[case].vmax) & (vm >= acopf_contingencies[case].vmin)).sum()/acopf_contingencies[case].nbus)*100
        dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
        # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())
    
        sff_rate_torch = ((Sff.real <= flow_max).sum()/acopf_contingencies[case].nl)*100        
        stt_rate_torch = ((Stt.real <= flow_max).sum()/acopf_contingencies[case].nl)*100        
        dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
        dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
        dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
        # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
        # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())
    
        dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())
    
        test_eq_real = test_eq_resid[:,:acopf_contingencies[case].nbus]
        test_eq_react = test_eq_resid[:,acopf_contingencies[case].nbus:]
        dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
        dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
        dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:acopf_contingencies[case].nbus] <= 1e-2) & (test_eq_resid[:,:acopf_contingencies[case].nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:acopf_contingencies[case].nbus].shape[1]*100).detach().cpu().numpy())
        dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,acopf_contingencies[case].nbus:] <= 1e-2) & (test_eq_resid[:,acopf_contingencies[case].nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,acopf_contingencies[case].nbus:].shape[1]*100).detach().cpu().numpy())

        print('Test batch {}: test loss {:.4f}, test obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}, p_g satisfication rate {:.4f}, q_g satisfication rate {:.4f}, v_m satisfication rate {:.4f}, test_line_limit_satisfication_rate {:.4f}, test_line_limit_satisfication_rate_Sf {:.4f}, test_line_limit_satisfication_rate_St {:.4f}, active eq satisfication rate {:.4f}, reactive eq satisfication rate {:.4f}'.format(
                    i, np.mean(test_stats['test_loss']), np.mean(test_stats['test_obj_cost']), np.mean(test_stats['test_ineq_max']), np.mean(test_stats['test_ineq_mean']),
                    np.mean(test_stats['test_ineq_q_g_num_viol_0']), np.mean(test_stats['test_ineq_v_m_num_viol_0']),
                    np.mean(test_stats['test_eq_max']), np.mean(test_stats['test_eq_mean']), np.mean(test_stats['test_p_g_satisfication rate (%)']), np.mean(test_stats['test_q_g_satisfication rate (%)']), np.mean(test_stats['test_v_m_satisfication rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']), np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']), np.mean(test_stats['test_active_eq_satisfication rate (%)']), np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))

    test_stats_contingency += [test_stats]
    print("\n")

Contingency  0
Test batch 0: test loss 11.4664, test obj 11.4214, ineq max 0.0211, ineq mean 0.0000, ineq q_g num viol 0.0000, ineq v_m num viol 7.0000, eq max 0.0002, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 100.0000, v_m satisfication rate 95.6790, test_line_limit_satisfication_rate 100.0000, test_line_limit_satisfication_rate_Sf 100.0000, test_line_limit_satisfication_rate_St 100.0000, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 1: test loss 11.4294, test obj 11.3769, ineq max 0.0213, ineq mean 0.0001, ineq q_g num viol 0.0000, ineq v_m num viol 6.5000, eq max 0.0002, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 100.0000, v_m satisfication rate 95.9877, test_line_limit_satisfication_rate 100.0000, test_line_limit_satisfication_rate_Sf 100.0000, test_line_limit_satisfication_rate_St 100.0000, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 

* Arithmetic mean

In [12]:
## Calculate the results of GraphLDE
for case in range(len(acopf_contingencies)):
    print("Contingency ", case)
    print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats_contingency[case]['test_obj_cost'])*10000, 4))
    print("GraphLDE eq. mean for test samples: ", np.mean(test_stats_contingency[case]['test_eq_mean']))
    print("GraphLDE eq. max for test samples: ", np.mean(test_stats_contingency[case]['test_eq_max']))
    print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats_contingency[case]['test_eq_real_mean']))
    print("GraphLDE eq. active max for test samples: ", np.mean(test_stats_contingency[case]['test_eq_real_max']))
    print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats_contingency[case]['test_eq_react_mean']))
    print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats_contingency[case]['test_eq_react_max']))
    
    print("\n")
    print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_mean']))
    print("GraphLDE ineq. max for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_max']))
    print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_p_g_mean']))
    print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_p_g_max']))
    print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_q_g_mean']))
    print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_q_g_max']))
    print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_v_m_mean']))
    print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_v_m_max']))
    print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_line_l_mean']))
    print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats_contingency[case]['test_ineq_line_l_max']))
    
    print("\n")
    print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats_contingency[case]['test_p_g_satisfication rate (%)']))
    print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats_contingency[case]['test_q_g_satisfication rate (%)']))
    print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats_contingency[case]['test_v_m_satisfication rate (%)']))
    print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats_contingency[case]['test_line_limit_satisfication_rate_Sf(%)']))
    print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats_contingency[case]['test_line_limit_satisfication_rate_St(%)']))
    print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats_contingency[case]['test_active_eq_satisfication rate (%)']))
    print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats_contingency[case]['test_reactive_eq_satisfication rate (%)']))
    
    print("\n")
    # print("DeepLDE time (ms) <== average value for test dataset:", (test_stats['time']/1000)*1e3)
    print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)
    print("\n")
    

Contingency  0
GraphLDE obj. value for test samples:  112600.34
GraphLDE eq. mean for test samples:  4.381728e-06
GraphLDE eq. max for test samples:  0.00015788498
GraphLDE eq. active mean for test samples:  1.7438695e-06
GraphLDE eq. active max for test samples:  3.2058993e-05
GraphLDE eq. reactive mean for test samples:  7.0195865e-06
GraphLDE eq. reactive max for test samples:  0.00015739084


GraphLDE ineq. mean for test samples:  0.000615523
GraphLDE ineq. max for test samples:  0.31409094
GraphLDE ineq. p_g mean for test samples:  0.0014134198
GraphLDE ineq. p_g max for test samples:  0.033922076
GraphLDE ineq. q_g mean for test samples:  0.00033377553
GraphLDE ineq. q_g max for test samples:  0.0061848164
GraphLDE ineq. v_m mean for test samples:  0.00021456662
GraphLDE ineq. v_m max for test samples:  0.03096992
GraphLDE ineq. line_l mean for test samples:  0.00079923467
GraphLDE ineq. line_l max for test samples:  0.2853413


GraphLDE p_g satisfication rate for test samples:  

* Harmonic mean

In [19]:
import statistics as st

## Calculate optimality gap
print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", st.harmonic_mean(test_stats['test_eq_mean'])) # print("LDF eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", st.harmonic_mean(test_stats['test_eq_max'])) # print("LDF eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", st.harmonic_mean(test_stats['test_eq_real_mean'])) # print("LDF eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", st.harmonic_mean(test_stats['test_eq_real_max'])) # print("LDF eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", st.harmonic_mean(test_stats['test_eq_react_mean'])) # print("LDF eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", st.harmonic_mean(test_stats['test_eq_react_max'])) # print("LDF eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))
print("\n")
print("GraphLDE ineq. mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_mean'])) # print("LDF ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", st.harmonic_mean(test_stats['test_ineq_max'])) # print("LDF ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_mean'])) # print("LDF ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_max'])) # print("LDF ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_mean'])) # print("LDF ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_max'])) # print("LDF ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_mean'])) # print("LDF ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_max'])) # print("LDF ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_mean'])) # print("LDF ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_max'])) # print("LDF ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))
print("\n")

print("GraphLDE p_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_p_g_satisfication rate (%)'].reshape(-1))) # print("LDF p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_q_g_satisfication rate (%)'].reshape(-1))) # print("LDF q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_v_m_satisfication rate (%)'].reshape(-1))) # print("LDF v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_St(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_active_eq_satisfication rate (%)'].reshape(-1))) # print("LDF active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_reactive_eq_satisfication rate (%)'].reshape(-1))) # print("LDF reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)


GraphLDE obj. value for test samples:  114521.91
GraphLDE eq. mean for test samples:  9.938747e-05
GraphLDE eq. max for test samples:  0.0027875048
GraphLDE eq. active mean for test samples:  7.965768e-05
GraphLDE eq. active max for test samples:  0.0027861511
GraphLDE eq. reactive mean for test samples:  0.000117550284
GraphLDE eq. reactive max for test samples:  0.0010674979


GraphLDE ineq. mean for test samples:  0.0
GraphLDE ineq. max for test samples:  0.0
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  0.0
GraphLDE ineq. q_g max for test samples:  0.0
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0
GraphLDE ineq. line_l max for test samples:  0.0


GraphLDE p_g satisfication rate for test samples:  95.96929
GraphLDE q_g satisfication rate for test samples:  86.06264
GraphLDE v_m satisfication rate f

/pscratch/sd/k/kjsong/conda/pytorch/2.8.0/lib/python3.12/statistics.py:593: RuntimeWarning: divide by zero encountered in scalar divide
  T, total, count = _sum(w / x if w else 0 for w, x in zip(weights, data))
